# Desabilitar pop-ups — Mobilidade Urbana — v1
1. No **app 3 (ExB)**: desliga o pop-up nos dois mapas (principal e concentração).
2. No **web map Mobilidade Urbana**: desliga o pop-up das camadas (o conteúdo do pop-up fica guardado, dá para religar depois).
Backup antes de tudo. Depois: **Ctrl+F5 → mudança pequena → Salvar → Publicar** (não precisa reiniciar o client).
Saída: `Documents\Saidas_Notebooks\Desabilitar_Popup_Mobilidade_v1.txt`

In [ ]:
# ===== CÉLULA 1 — Configuração =====
import os, json, datetime, shutil
NB_NOME = "Desabilitar_Popup_Mobilidade_v1"
CFG_APP = r"C:\ExB_1.19\server\public\apps\3\resources\config\config.json"
WEBMAP_ID = "c80cb05e18a645609f8f84740078b859"
MAPAS = ["widget_mapa", "widget_mapa_mini"]
home = os.path.expanduser("~")
_c = [os.path.join(home, "Documents", "Saidas_Notebooks"), os.path.join(home, "OneDrive", "Documents", "Saidas_Notebooks")]
PASTA_SAIDA = next((c for c in _c if os.path.isdir(c)), _c[0]); os.makedirs(PASTA_SAIDA, exist_ok=True)
PASTA_BKP = os.path.join(PASTA_SAIDA, f"backup_popup_mobilidade_{datetime.datetime.now():%Y%m%d_%H%M%S}"); os.makedirs(PASTA_BKP, exist_ok=True)
ARQ_TXT = os.path.join(PASTA_SAIDA, NB_NOME + ".txt")
with open(ARQ_TXT, "w", encoding="utf-8") as f:
    f.write(f"{NB_NOME} — executado em {datetime.datetime.now():%d/%m/%Y %H:%M:%S}\n\n")
def log(*a):
    s = " ".join(str(x) for x in a); print(s)
    with open(ARQ_TXT, "a", encoding="utf-8") as f: f.write(s + "\n")
log("▶ CÉLULA 1 — Configuração: iniciando...")
log("  Backups:", PASTA_BKP)
log("✔ CÉLULA 1 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 2 — App 3: desligar pop-up nos mapas =====
log("\n▶ CÉLULA 2 — Pop-up nos mapas do app 3: iniciando...")
shutil.copy2(CFG_APP, os.path.join(PASTA_BKP, "app3_config_antes.json"))
with open(CFG_APP, encoding="utf-8") as f:
    cfg = json.load(f)
for wid in MAPAS:
    w = cfg["widgets"].get(wid)
    if not w:
        log("  ! widget não encontrado:", wid); continue
    w.setdefault("config", {})["disablePopUp"] = True
    log("  · pop-up desligado em:", w.get("label", wid))
cfg["timestamp"] = int(datetime.datetime.now().timestamp() * 1000)
with open(CFG_APP, "w", encoding="utf-8") as f:
    json.dump(cfg, f, ensure_ascii=False, indent=2)
log("✔ CÉLULA 2 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 3 — Web map: desligar pop-up das camadas =====
log("\n▶ CÉLULA 3 — Pop-up no web map: iniciando...")
from arcgis.gis import GIS
gis = GIS("pro")
item = gis.content.get(WEBMAP_ID)
wm = item.get_data()
with open(os.path.join(PASTA_BKP, "webmap_antes.json"), "w", encoding="utf-8") as f:
    json.dump(wm, f, ensure_ascii=False, indent=2)
for ol in wm.get("operationalLayers", []):
    ol["disablePopup"] = True
    log("  · pop-up desligado:", ol.get("title"))
item.update(item_properties={"text": json.dumps(wm)})
log("✔ CÉLULA 3 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 4 — Próximos passos e links =====
log("\n▶ CÉLULA 4 — Links: iniciando...")
log("  Construtor: Ctrl+F5 → mudança pequena → Salvar → Publicar (não precisa reiniciar o client)")
log("   Construtor (app 3): https://localhost:3001/builder/?id=3")
log("   Prévia local:       https://localhost:3001/experience/3/?locale=pt-br")
log("   Web map:            https://geo.apucarana.pr.gov.br/portal/apps/mapviewer/index.html?webmap=" + WEBMAP_ID)
log(f"  Backups em: {PASTA_BKP}")
log(f"  Saída completa: {ARQ_TXT}")
log("✔ CÉLULA 4 concluída com sucesso.")
